# Pandas va Sklearn: Gipotezalarni tekshirish va Unknown Values bilan ishlash

Bu notebookda:

1. `sklearn.datasets`dan dataset topamiz va statistik **gipotezani** t-test yordamida tekshiramiz
2. Yana bir sklearn dataseti (`load_wine`) ustida ham gipoteza tekshiramiz
3. **Unknown / missing values**ni sun'iy yaratib, ularni aniqlash va tozalash usullarini ko'ramiz

---

## 1. Kutubxonalarni import qilish

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.datasets import load_diabetes, load_wine

pd.set_option('display.max_columns', 20)

## 2. Sklearn'dan dataset topish: Diabetes dataseti

`load_diabetes()` — 442 nafar bemorning yoshi, jinsi, BMI, qon bosimi va boshqa ko'rsatkichlarini o'z ichiga oladi. Maqsad — kasallik progressini (`target`) bashorat qilish.

In [4]:
data = load_diabetes()
diab_df = pd.DataFrame(data.data, columns=data.feature_names)
diab_df['target'] = data.target

print(diab_df.shape)
diab_df.head()

(442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 3. Gipotezani shakllantirish (Diabetes dataseti)

**Savol:** BMI (tana vazni indeksi) o'rtacha darajadan yuqori va past bo'lgan bemorlar orasida kasallik progressi (`target`) statistik jihatdan farq qiladimi?

* **H0:** Ikki guruh o'rtasida `target` bo'yicha farq yo'q
* **H1:** Ikki guruh o'rtasida statistik ahamiyatli farq bor

In [5]:
bmi_median = diab_df['bmi'].median()

high_bmi = diab_df[diab_df['bmi'] > bmi_median]['target']
low_bmi = diab_df[diab_df['bmi'] <= bmi_median]['target']

t_stat, p_value = stats.ttest_ind(high_bmi, low_bmi, equal_var=False)

print(f'Yuqori BMI guruhi o\'rtachasi: {high_bmi.mean():.2f}')
print(f'Past BMI guruhi o\'rtachasi: {low_bmi.mean():.2f}')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.10f}')

alpha = 0.05
if p_value < alpha:
    print('H0 rad etildi: BMI darajasi kasallik progressiga statistik ta\'sir qiladi.')
else:
    print('H0 rad etilmadi: statistik ahamiyatli farq topilmadi.')

Yuqori BMI guruhi o'rtachasi: 191.46
Past BMI guruhi o'rtachasi: 113.52
t-statistic: 12.2750
p-value: 0.0000000000
H0 rad etildi: BMI darajasi kasallik progressiga statistik ta'sir qiladi.


---

## 4. Yana bir sklearn dataseti: Wine dataseti bilan gipoteza

`load_wine()` — 178 ta vino namunasining kimyoviy tarkibi (alcohol, magnesium, flavanoids va h.k.) va ular qaysi sinfga (`target`) tegishli ekanligini o'z ichiga oladi.

**Savol:** 0-sinf va 1-sinf vinolar orasida `alcohol` darajasi statistik jihatdan farq qiladimi?

* **H0:** Ikki sinf o'rtasida `alcohol` bo'yicha farq yo'q
* **H1:** Ikki sinf o'rtasida statistik ahamiyatli farq bor

In [6]:
wine = load_wine()
wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
wine_df['target'] = wine.target

class0 = wine_df[wine_df['target'] == 0]['alcohol']
class1 = wine_df[wine_df['target'] == 1]['alcohol']

print('0-sinf soni:', len(class0))
print('1-sinf soni:', len(class1))

t_stat, p_value = stats.ttest_ind(class0, class1, equal_var=False)

print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.10f}')

if p_value < alpha:
    print('H0 rad etildi: sinflar orasida alcohol darajasi statistik jihatdan farq qiladi.')
else:
    print('H0 rad etilmadi: statistik ahamiyatli farq topilmadi.')

0-sinf soni: 59
1-sinf soni: 71
t-statistic: 16.7113
p-value: 0.0000000000
H0 rad etildi: sinflar orasida alcohol darajasi statistik jihatdan farq qiladi.


---

## 5. Unknown / Missing Values yaratish va aniqlash

`load_wine()` dataseti toza (bo'sh qiymatlarsiz), shuning uchun buni ko'rsatish uchun sun'iy ravishda `NaN` va `'unknown'` qiymatlar qo'shamiz.

In [7]:
wine_missing = wine_df.copy()

wine_missing['proline_level'] = pd.cut(
    wine_missing['proline'],
    bins=3,
    labels=['past', "o'rta", 'yuqori']
).astype(str)

np.random.seed(42)
rows_alcohol = wine_missing.sample(frac=0.05, random_state=42).index
wine_missing.loc[rows_alcohol, 'alcohol'] = np.nan

rows_level = wine_missing.sample(frac=0.04, random_state=7).index
wine_missing.loc[rows_level, 'proline_level'] = 'unknown'

wine_missing[['alcohol', 'proline_level']].head(10)

,alcohol,proline_level
0,14.23,o'rta
1,13.20,o'rta
2,13.16,o'rta
3,14.37,yuqori
4,13.24,past
5,14.20,yuqori
6,14.39,yuqori
7,14.06,yuqori
8,14.83,o'rta
9,13.86,o'rta


### 5.1. Noma'lum belgilarni haqiqiy NaN'ga aylantirish

In [8]:
wine_missing.replace('unknown', np.nan, inplace=True)

missing_percent = wine_missing.isnull().mean() * 100
missing_percent[missing_percent > 0]

alcohol          5.056180
proline_level    3.932584
dtype: float64

### 5.2. Missing qiymatlarni tozalash usullari

1. **dropna** — qatorlarni o'chirish
2. **fillna (median)** — sonli ustunlar uchun
3. **fillna (mode)** — kategoriyal ustunlar uchun

In [9]:
wine_dropped = wine_missing.dropna(subset=['alcohol', 'proline_level'])
print("Dropna'dan keyingi shakl:", wine_dropped.shape, '| Asl shakl:', wine_missing.shape)

Dropna'dan keyingi shakl: (162, 15) | Asl shakl: (178, 15)


In [10]:
wine_filled = wine_missing.copy()

wine_filled['alcohol'] = wine_filled['alcohol'].fillna(wine_filled['alcohol'].median())
wine_filled['proline_level'] = wine_filled['proline_level'].fillna(wine_filled['proline_level'].mode()[0])

print(wine_filled[['alcohol', 'proline_level']].isnull().sum())

alcohol          0
proline_level    0
dtype: int64


### 5.3. To'ldirishdan oldin va keyin o'rtachani solishtirish

In [11]:
print(f"Asl alcohol o'rtachasi: {wine_df['alcohol'].mean():.4f}")
print(f"To'ldirilgandan keyingi o'rtacha: {wine_filled['alcohol'].mean():.4f}")

Asl alcohol o'rtachasi: 13.0006
To'ldirilgandan keyingi o'rtacha: 12.9965


---

## 6. Xulosa

* Gipotezalarni tekshirishda **t-test** yordamida ikki guruh o'rtasidagi farq statistik ahamiyatga ega yoki emasligini aniqlaymiz (p-value < 0.05 bo'lsa H0 rad etiladi).
* Unknown qiymatlar (`'?'`, `'unknown'` va h.k.) avval haqiqiy `NaN`ga aylantirilishi kerak, so'ng `dropna` yoki `fillna` orqali tozalanadi.
* Median bilan to'ldirish odatda o'rtacha qiymatni sezilarli o'zgartirmaydi, shu bilan birga ma'lumot yo'qotilmaydi.